In [1]:
a = 5

In [2]:
import mysql.connector
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()


# =========================
# MYSQL CONNECTION
# =========================

def get_db_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="data_science",
        port=3306
    )


# =========================
# GENERIC QUERY FUNCTION
# =========================

def execute_query(query):
    db = None
    cursor = None

    try:
        db = get_db_connection()

        if not db.is_connected():
            return "MySQL connection failed."

        cursor = db.cursor()

        cursor.execute(query)

        data = cursor.fetchall()

        return data

    except mysql.connector.Error as e:
        return f"MySQL Error: {e}"

    finally:

        if cursor is not None:
            cursor.close()

        if db is not None and db.is_connected():
            db.close()


# =========================
# TOOLS
# =========================

@tool
def get_student_data():
    """Get all student records from student_info_ai."""

    return execute_query(
        "SELECT * FROM student_info_ai"
    )


@tool
def get_employee_data():
    """Get all employee records from employee1."""

    return execute_query(
        "SELECT * FROM employee1"
    )


@tool
def get_department_data():
    """Get all department records from departments1."""

    return execute_query(
        "SELECT * FROM departments1"
    )


@tool
def get_product_data():
    """Get all product records from products."""

    return execute_query(
        "SELECT * FROM products"
    )


@tool
def get_order_data():
    """Get all order records from orders."""

    return execute_query(
        "SELECT * FROM orders"
    )


# =========================
# TEST MYSQL
# =========================

test_db = None

try:

    test_db = get_db_connection()

    print("MySQL connected:", test_db.is_connected())

finally:

    if test_db is not None and test_db.is_connected():
        test_db.close()


# =========================
# TEST STUDENT TOOL
# =========================

print("\nStudent data:")

print(
    get_student_data.invoke({})
)


# =========================
# GEMINI
# =========================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)


# =========================
# TOOLS LIST
# =========================

tools = [
    get_student_data,
    get_employee_data,
    get_department_data,
    get_product_data,
    get_order_data
]


# =========================
# CREATE AGENT
# =========================

agent = create_agent(

    model=llm,

    tools=tools,

    system_prompt="""

You are a database assistant.

You answer questions using data retrieved from the MySQL database.

Use the correct tool according to the user's question.

Student questions:
Use get_student_data.

Employee questions:
Use get_employee_data.

Department questions:
Use get_department_data.

Product questions:
Use get_product_data.

Order questions:
Use get_order_data.

Never invent database information.

Only use information returned by the database.

Analyze the returned records and provide a clear answer.

"""
)

print("\nAgent created successfully.")

MySQL connected: True

Student data:
[('Aarav Patel', 'BSc IT', 85), ('Meera Shah', 'MSc IT', 92), ('Rohan Mehta', 'BCA', 78), ('Priya Desai', 'BSc Computer Science', 88), ('Karan Joshi', 'MCA', 74), ('Neha Trivedi', 'MSc Data Science', 95), ('Vivek Shah', 'BCA', 81), ('Ananya Patel', 'BSc IT', 89), ('Rahul Desai', 'MCA', 76), ('Sneha Mehta', 'MSc IT', 91)]

Agent created successfully.


In [6]:
question = input("Ask your question: ")

result = agent.invoke({
    "messages": [
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'I am a database assistant and can only answer questions using information retrieved from the database regarding students, employees, departments, products, and orders. \n\nSince your question is about general knowledge and not present in the database, I cannot answer it.', 'extras': {'signature': 'EqoRCqcRARFNMg/BSgu0fkZ59PJV2zD6xf2qwcTYg7I30Hs8H8zyTmGrk2cnqm5X5KE3e1fPZ7Z71hJ0XaOJyufhralE3BBgrilkbeu8FohDW/cwoAMr2MRu1hDtOzONOIlpn4OySrAf7tY5mpSpLoxmzsxYsUM3p2HILIXTx5D9Zj2wmE3wJWxre7FK9xGXmaUPPoSLFzTQwtJx2xTBJ5oBjzbBVnq7bcBwjEuWm2rYK25LDnHU5xOHNR4WdRlnxD6zO3Df8l9/jArvwytHyPJoU7qUYD4h/8+0ZT1uNP7mxho4usVSN7jA3qnFBf80vQHZ+VaVbkx8inLXTH+LLcTGbUeU4XoV+UyZTomin9De91e/hrQDhHlbjZ0HCQUR0Y/DwnagQWrhlZ6hXG11ijX/iJs9HJ0IwTVg+ROnLoaHUP53XTerugSm7DV6bv0xCEQDJhM+0Jj7pEnokyaltslfC5Og5QlKdilTQYI3CWvuBehLBBlfk9PGKqgNvXod+9GXDPh5J3RS5+awq5fZZlFmWTk72Kl0dkC8cuXR751uHbQnwaXk99vba/PPs+VphXGekEfSO/KAqKJqvRruUPV19S4on6A2PkOsJ7nbqQ9lSeMhwffB5PTJOVFuJRQHfNTZTLQsmnKBidBU0uLABaU21Zq21l9FFs